# Import RDKit, install if not available

In [ ]:

try:
  import rdkit
except ImportError:
  !pip install rdkit
  import rdkit

import warnings

# In the list add proper known monomners for testing reactions

In [ ]:
sample_smiles = [
    "CC(C)(c1ccc(OCC2CO2)cc1)c1ccc(OCC2CO2)cc1",   # [0] DGEBA (epoxide)
    "NCCN",                                         # [1] ethylenediamine (primary amine)
]

# Validate SMILES strings

In [ ]:
from rdkit import Chem
for s in sample_smiles:
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        raise ValueError(f"SMILES '{s}' is invalid (RDKit could not parse it).")


# Define your required monomers functionality 

In [ ]:
monomers = {
    'epoxide_monomer': {
        'functionality_type': 'di_identical',
        'smarts_1': '[CH2;r3]1[O;r3][CH;r3]1',
        'group_name': 'epoxide',
        'comments': 'Terminal (glycidyl) epoxide'
    },
    'primary_amine_monomer': {
        'functionality_type': 'di_identical',
        'smarts_1': '[NX3;H2;!$(NC=O)]',
        'group_name': 'primary_amine',
        'comments': None
    },
}

# Run validation checks on monomer functionality definitions

In [ ]:

for funct in monomers.values():
    functionality_type = funct["functionality_type"]
    if functionality_type is None:
        raise ValueError("Functionality type is missing.")
    if functionality_type not in ["di_identical", "di_different", "mono", "vinyl"]:
        raise ValueError(f"Functionality type '{functionality_type}' is invalid.")
    if "smarts_1" not in funct:
        raise ValueError("Missing 'smarts_1' must be provided.")
    smart_1 = funct.get("smarts_1")
    if Chem.MolFromSmarts(smart_1) is None:
        raise ValueError(f"'smarts_1' '{smart_1}' is invalid.")
    if functionality_type == "di_different":
        smart_2 = funct.get("smarts_2")
        if smart_2 is None:
            raise ValueError("Missing 'smarts_2' for 'di_different' functionality type.")
        if Chem.MolFromSmarts(smart_2) is None:
            raise ValueError(f"'smarts_2' '{smart_2}' is invalid.")
    if "group_name" not in funct:
        raise ValueError("Missing 'group_name' must be provided.")
    group_name = funct.get("group_name")
    if not isinstance(group_name, str) or not group_name.strip():
        raise ValueError(f"'group_name' '{group_name}' is invalid.")
    print(f"{group_name} definitions are valid.")
    

# Testing Function Block

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

def validate_functional_group(sample_smiles: list[str], monomers: dict[str, dict[str, str]]):
    results = []

    for monomer_1_smiles in sample_smiles:
        mol_1 = Chem.MolFromSmiles(monomer_1_smiles)
        if mol_1 is None:
            print(f"Skipping invalid SMILES: {monomer_1_smiles}")
            continue

        # make explicit Hs (MUST assign)
        mol_1 = Chem.AddHs(mol_1)

        for funtionality in monomers.values():

            if funtionality["functionality_type"] == "di_different":
                smarts_1 = Chem.MolFromSmarts(funtionality["smarts_1"])
                smarts_2 = Chem.MolFromSmarts(funtionality["smarts_2"])
                if smarts_1 is None or smarts_2 is None:
                    continue

                if mol_1.HasSubstructMatch(smarts_1) and mol_1.HasSubstructMatch(smarts_2):
                    print(f"Monomer {monomer_1_smiles} matches functionality {funtionality['group_name']}")
                    print("Different functional passed.")

                    # get *all* matches (not just first) so highlight is complete
                    matches_1 = mol_1.GetSubstructMatches(smarts_1)
                    matches_2 = mol_1.GetSubstructMatches(smarts_2)

                    atoms_to_highlight = []
                    for m in matches_1:
                        atoms_to_highlight.extend(list(m))
                    for m in matches_2:
                        atoms_to_highlight.extend(list(m))

                    # dedupe + keep order-ish
                    atoms_to_highlight = list(dict.fromkeys(atoms_to_highlight))

                    results.append((mol_1, atoms_to_highlight, funtionality["group_name"]))

            if funtionality["functionality_type"] == "di_identical":
                smarts_1 = Chem.MolFromSmarts(funtionality["smarts_1"])
                if smarts_1 is None:
                    continue

                matches = mol_1.GetSubstructMatches(smarts_1)
                if len(matches) >= 2:
                    print(f"Monomer {monomer_1_smiles} matches functionality {funtionality['group_name']}")
                    print("Identical functional passed.")

                    atoms_to_highlight = []
                    for match in matches:
                        atoms_to_highlight.extend(list(match))
                    atoms_to_highlight = list(dict.fromkeys(atoms_to_highlight))

                    results.append((mol_1, atoms_to_highlight, funtionality["group_name"]))

            if funtionality["functionality_type"] in {"vinyl", "mono"}:
                smarts_1 = Chem.MolFromSmarts(funtionality["smarts_1"])

                if smarts_1 is None:
                    continue

                matches = mol_1.GetSubstructMatches(smarts_1)

                if matches:
                    print(
                        f"Monomer {monomer_1_smiles} matches functionality "
                        f"{funtionality['group_name']}"
                    )
                    print(
                        f"{funtionality['functionality_type'].capitalize()} "
                        "functional passed."
                    )

                    atoms_to_highlight = list(
                        dict.fromkeys(
                            atom_idx
                            for match in matches
                            for atom_idx in match
                        )
                    )

                    results.append(
                        (
                            mol_1,
                            atoms_to_highlight,
                            funtionality["group_name"],
                        )
                    )
                    
    if not results:
        raise ValueError("No matching functional group found.")
    return results


# Test the function and Display results

In [ ]:


results = validate_functional_group(sample_smiles, monomers)


# Create a grid from the molecules and highlights directly
grid_img = Draw.MolsToGridImage(
    [r[0] for r in results], 
    molsPerRow=3, 
    subImgSize=(900, 900),
    highlightAtomLists=[r[1] for r in results],
    legends=[r[2] for r in results]
)

display(grid_img)

# Define reaction definitions

In [ ]:
reactions_definitions = {
    'Epoxy + primary amine': {
        'same_reactants': False,
        'reactant_1': 'epoxide',
        'reactant_2': 'primary_amine',
        'product': 'secondary_amine',
        'delete_atom': False,
        'reaction': (
            '[CH2:1]1[O:3][CH:4]1.'
            '[NX3;H2;!$(NC=O):2]-[#1:5]'
            '>>'
            '[N:2]-[C:1]-[C:4]-[O:3]-[#1:5]'
        ),
    },
}

# Validate reaction SMARTS and references

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdChemReactions
import json
import warnings


def validate_reaction_smarts(smarts_string):
    """
    Validate the syntax of a reaction SMARTS string using RDKit.
    Returns True if valid and False otherwise.
    """
    try:
        rxn = rdChemReactions.ReactionFromSmarts(smarts_string)

        if rxn is not None:
            print(f"'{smarts_string}' is a valid reaction SMARTS.")
            return True

        print(
            f"'{smarts_string}' is an invalid reaction SMARTS "
            "(ReactionFromSmarts returned None)."
        )
        return False

    except Exception as e:
        print(f"'{smarts_string}' is an invalid reaction SMARTS.")
        print(f"Error details: {e}")
        return False


def are_all_atoms_mapped(reaction_smarts):
    """
    Check that every atom in the reactants and products has an atom map number.
    """
    rxn = rdChemReactions.ReactionFromSmarts(reaction_smarts)

    if rxn is None:
        return False, "Invalid reaction SMARTS"

    # Check reactants
    for reactant_mol in rxn.GetReactants():
        for atom in reactant_mol.GetAtoms():
            if atom.GetAtomMapNum() == 0:
                return False, "Unmapped atom found in reactants"

    # Check products
    for product_mol in rxn.GetProducts():
        for atom in product_mol.GetAtoms():
            if atom.GetAtomMapNum() == 0:
                return False, "Unmapped atom found in products"

    return True, "All atoms are mapped"


def validate_initiator_atom_maps(reaction_smarts, initiator_atom_maps=(1, 2)):
    rxn = rdChemReactions.ReactionFromSmarts(reaction_smarts)
    if rxn is None:
        return False, "Invalid reaction SMARTS"
    m1, m2 = int(initiator_atom_maps[0]), int(initiator_atom_maps[1])
    if m1 <= 0 or m2 <= 0 or m1 == m2:
        return False, "Initiator atom maps must be two different positive numbers."

    initiator_locations = {m1: [], m2: []}
    for reactant_index, reactant_mol in enumerate(rxn.GetReactants(), start=1):
        for atom in reactant_mol.GetAtoms():
            map_num = atom.GetAtomMapNum()
            if map_num in (m1, m2):
                initiator_locations[map_num].append(reactant_index)

    missing_maps = [m for m in (m1, m2) if len(initiator_locations[m]) == 0]
    if missing_maps:
        return False, f"Missing initiator atom map(s): {missing_maps}"
    for m in (m1, m2):
        if len(initiator_locations[m]) > 1:
            return False, f"Initiator atom map {m} appears more than once in the reactant SMARTS."

    r1, r2 = initiator_locations[m1][0], initiator_locations[m2][0]
    if r1 == r2:
        return False, f"Initiator atom maps {m1} and {m2} must belong to different reactant molecules."
    return True, (f"Initiator atom maps are valid: map {m1} is in reactant {r1} "
                  f"and map {m2} is in reactant {r2}.")


for reaction_name, reaction in reactions_definitions.items():

    print(f"\nValidating reaction: {reaction_name}")

    reaction_smarts = reaction.get("reaction")
    same_reactants = reaction.get("same_reactants")
    reactant_1 = reaction.get("reactant_1")
    product = reaction.get("product")
    delete_atom = reaction.get("delete_atom")
    reference = reaction.get("reference")

    # Validate required reaction-definition fields
    if same_reactants is None:
        raise ValueError(
            f"{reaction_name}: 'same_reactants' flag is missing."
        )

    if reactant_1 is None:
        raise ValueError(
            f"{reaction_name}: 'reactant_1' information is missing."
        )

    if not same_reactants:
        reactant_2 = reaction.get("reactant_2")

        if reactant_2 is None:
            raise ValueError(
                f"{reaction_name}: 'reactant_2' information is missing "
                "for a reaction with different reactants."
            )

    if product is None:
        raise ValueError(
            f"{reaction_name}: 'product' information is missing."
        )

    if delete_atom is None:
        raise ValueError(
            f"{reaction_name}: 'delete_atom' flag is missing."
        )

    if reaction_smarts is None:
        raise ValueError(
            f"{reaction_name}: reaction SMARTS is missing."
        )

    # Validate reaction SMARTS syntax
    if not validate_reaction_smarts(reaction_smarts):
        raise ValueError(
            f"{reaction_name}: invalid reaction SMARTS."
        )

    # Check that every atom is mapped
    is_mapped, mapping_message = are_all_atoms_mapped(
        reaction_smarts
    )

    if not is_mapped:
        warnings.warn(
            f"{reaction_name}: {mapping_message}"
        )
    else:
        print(mapping_message)

    # Check AutoREACTER initiator atom maps 1 and 2
    initiators_valid, initiator_message = validate_initiator_atom_maps(
        reaction_smarts
    )

    if not initiators_valid:
        raise ValueError(
            f"{reaction_name}: {initiator_message}"
        )

    print(initiator_message)

    # Print validated reaction definition
    print(json.dumps(reaction, indent=2))

    print(f"{reaction_name}: reaction definition is valid.")

print("\nAll reaction definitions are valid.")

In [ ]:
from collections import Counter


def smart_mapping(reactant, smarts_template, match_tuple):
    """
    Transfer atom-map numbers from a SMARTS template to a matched reactant.

    Only explicitly mapped SMARTS atoms are processed. Unmapped atoms are
    allowed and remain unmapped.

    Mapping rules
    -------------
    - Atom-map number 0 means "unmapped" and is ignored.
    - Every nonzero map number in the SMARTS template must be unique.
    - Each mapped SMARTS atom must correspond to exactly one matched atom
      in the reactant.
    - Existing atom-map numbers on the matched reactant atoms are replaced
      by the mapping defined by the SMARTS template.

    Parameters
    ----------
    reactant : Chem.Mol
        Reactant molecule to modify in place.

    smarts_template : Chem.Mol
        SMARTS reactant template containing atom-map numbers.

    match_tuple : tuple
        RDKit substructure match. Position i corresponds to SMARTS atom i.

    Raises
    ------
    ValueError
        If a mapped SMARTS atom has no corresponding matched atom or if
        duplicate atom-map numbers occur in the SMARTS template.
    """
    if not match_tuple:
        return

    map_numbers = [
        atom.GetAtomMapNum()
        for atom in smarts_template.GetAtoms()
        if atom.GetAtomMapNum() != 0
    ]

    duplicate_maps = sorted(
        atom_map
        for atom_map, count in Counter(map_numbers).items()
        if count > 1
    )

    if duplicate_maps:
        raise ValueError(
            "SMARTS template contains duplicate atom-map numbers: "
            f"{duplicate_maps}. "
            "Every mapped atom must have a unique map number."
        )

    for smarts_atom in smarts_template.GetAtoms():
        map_num = smarts_atom.GetAtomMapNum()

        if map_num == 0:
            continue

        smarts_index = smarts_atom.GetIdx()

        if smarts_index >= len(match_tuple):
            raise ValueError(
                f"SMARTS atom index {smarts_index} with map :{map_num} "
                "has no corresponding atom in the RDKit match."
            )

        reactant_atom_index = match_tuple[smarts_index]

        reactant_atom = reactant.GetAtomWithIdx(
            reactant_atom_index
        )

        reactant_atom.SetAtomMapNum(map_num)


def _atom_map_counts(templates) -> Counter:
    """
    Count all nonzero atom-map numbers in a collection of RDKit templates.

    Unmapped atoms (map number 0) are ignored.
    """
    counts = Counter()

    for template in templates:
        for atom in template.GetAtoms():
            atom_map = atom.GetAtomMapNum()

            if atom_map:
                counts[atom_map] += 1

    return counts


def _atom_maps_in_templates(templates) -> set[int]:
    """
    Return all nonzero atom-map numbers present in RDKit templates.
    """
    return set(_atom_map_counts(templates))


def _duplicate_atom_maps(templates) -> list[int]:
    """
    Return atom-map numbers appearing more than once in a template set.

    For AutoREACTER, each nonzero atom-map number must identify exactly
    one atom on each side of the reaction.
    """
    counts = _atom_map_counts(templates)

    return sorted(
        atom_map
        for atom_map, count in counts.items()
        if count > 1
    )


def _has_bond_between_atom_maps(
    templates,
    atom_map_1: int,
    atom_map_2: int,
) -> bool:
    """
    Return True if any template contains a bond between two atom maps.
    """
    target = {atom_map_1, atom_map_2}

    for template in templates:
        for bond in template.GetBonds():
            begin_map = (
                bond.GetBeginAtom().GetAtomMapNum()
            )

            end_map = (
                bond.GetEndAtom().GetAtomMapNum()
            )

            if {begin_map, end_map} == target:
                return True

    return False


def _validate_reaction_smarts(
    reaction_name: str,
    reaction: dict,
) -> list[str]:
    """
    Validate one AutoREACTER reaction-library entry.

    AutoREACTER atom-mapping rules
    ------------------------------
    Not every atom must be mapped.

    However, every atom that IS mapped must obey a strict 1:1 mapping:

        reactant atom :N  <->  product atom :N

    Therefore:

        - each nonzero atom-map number may occur only once across all
          reactant templates;
        - each nonzero atom-map number may occur only once across all
          product templates;
        - the mapped-atom set on the reactant side must exactly match
          the mapped-atom set on the product side.

    AutoREACTER additionally requires initiator atoms :1 and :2 by
    default. Both must exist on both sides of the reaction, and the
    product must contain a bond between them.

    Optional override
    -----------------
    A different initiator pair may be specified with:

        "initiator_atom_maps": (a, b)

    Special reactions may disable the initiator-bond requirement with:

        "validate_initiator_bond": False

    Note
    ----
    Disabling the initiator-bond check does NOT disable the general
    1:1 atom-mapping validation.
    """
    errors: list[str] = []

    smarts = reaction.get("reaction")

    if not isinstance(smarts, str) or not smarts.strip():
        return [
            f"{reaction_name}: missing or invalid required key "
            "'reaction'"
        ]

    if rdChemReactions is None:
        return [
            f"{reaction_name}: RDKit is required to validate "
            "reaction SMARTS"
        ]

    initiator_atom_maps = reaction.get(
        "initiator_atom_maps",
        (1, 2),
    )

    if (
        not isinstance(
            initiator_atom_maps,
            (list, tuple),
        )
        or len(initiator_atom_maps) != 2
    ):
        return [
            f"{reaction_name}: 'initiator_atom_maps' must contain "
            "exactly two atom-map numbers"
        ]

    try:
        initiator_1 = int(initiator_atom_maps[0])
        initiator_2 = int(initiator_atom_maps[1])

    except (TypeError, ValueError):
        return [
            f"{reaction_name}: 'initiator_atom_maps' must contain "
            "integer atom-map numbers"
        ]

    if initiator_1 <= 0 or initiator_2 <= 0:
        return [
            f"{reaction_name}: initiator atom-map numbers must be "
            "greater than zero"
        ]

    if initiator_1 == initiator_2:
        return [
            f"{reaction_name}: initiator atom maps must be different"
        ]

    try:
        rdkit_reaction = (
            rdChemReactions.ReactionFromSmarts(smarts)
        )

    except Exception as error:
        return [
            f"{reaction_name}: invalid reaction SMARTS: {error}"
        ]

    if rdkit_reaction is None:
        return [
            f"{reaction_name}: RDKit could not parse reaction SMARTS"
        ]

    num_reactants = (
        rdkit_reaction.GetNumReactantTemplates()
    )

    num_products = (
        rdkit_reaction.GetNumProductTemplates()
    )

    if num_reactants != 2:
        errors.append(
            f"{reaction_name}: AutoREACTER currently requires "
            f"exactly 2 reactant templates; found {num_reactants}"
        )

    if num_products == 0:
        errors.append(
            f"{reaction_name}: reaction SMARTS contains "
            "no product templates"
        )

    reactant_templates = [
        rdkit_reaction.GetReactantTemplate(i)
        for i in range(num_reactants)
    ]

    product_templates = [
        rdkit_reaction.GetProductTemplate(i)
        for i in range(num_products)
    ]

    # ---------------------------------------------------------
    # Validate uniqueness of mapped atoms
    # ---------------------------------------------------------

    duplicate_reactant_maps = _duplicate_atom_maps(
        reactant_templates
    )

    duplicate_product_maps = _duplicate_atom_maps(
        product_templates
    )

    if duplicate_reactant_maps:
        errors.append(
            f"{reaction_name}: duplicate atom-map numbers in "
            f"reactants: {duplicate_reactant_maps}. "
            "Each mapped atom must have a unique map number."
        )

    if duplicate_product_maps:
        errors.append(
            f"{reaction_name}: duplicate atom-map numbers in "
            f"products: {duplicate_product_maps}. "
            "Each mapped atom must have a unique map number."
        )

    # ---------------------------------------------------------
    # Get mapped-atom sets
    # ---------------------------------------------------------

    reactant_maps = _atom_maps_in_templates(
        reactant_templates
    )

    product_maps = _atom_maps_in_templates(
        product_templates
    )

    # ---------------------------------------------------------
    # Require strict 1:1 mapping
    # ---------------------------------------------------------

    missing_from_products = (
        reactant_maps - product_maps
    )

    missing_from_reactants = (
        product_maps - reactant_maps
    )

    if missing_from_products:
        errors.append(
            f"{reaction_name}: mapped atoms present in reactants "
            "but missing from products: "
            f"{sorted(missing_from_products)}. "
            "Mapped atoms must have a 1:1 reactant/product mapping."
        )

    if missing_from_reactants:
        errors.append(
            f"{reaction_name}: mapped atoms present in products "
            "but missing from reactants: "
            f"{sorted(missing_from_reactants)}. "
            "Mapped atoms must have a 1:1 reactant/product mapping."
        )

    # ---------------------------------------------------------
    # Validate mandatory initiator atoms
    # ---------------------------------------------------------

    required_maps = {
        initiator_1,
        initiator_2,
    }

    missing_reactant_initiators = (
        required_maps - reactant_maps
    )

    missing_product_initiators = (
        required_maps - product_maps
    )

    if missing_reactant_initiators:
        errors.append(
            f"{reaction_name}: required initiator atom maps "
            "missing from reactants: "
            f"{sorted(missing_reactant_initiators)}"
        )

    if missing_product_initiators:
        errors.append(
            f"{reaction_name}: required initiator atom maps "
            "missing from products: "
            f"{sorted(missing_product_initiators)}"
        )

    # ---------------------------------------------------------
    # Validate required product initiator bond
    # ---------------------------------------------------------

    validate_initiator_bond = reaction.get(
        "validate_initiator_bond",
        True,
    )

    if (
        validate_initiator_bond
        and not missing_product_initiators
    ):
        product_has_initiator_bond = (
            _has_bond_between_atom_maps(
                product_templates,
                initiator_1,
                initiator_2,
            )
        )

        if not product_has_initiator_bond:
            errors.append(
                f"{reaction_name}: product does not contain "
                "the required AutoREACTER initiator bond between "
                f"atom maps :{initiator_1} and :{initiator_2}"
            )

    return errors


def validate_reactions(reactions: dict) -> None:
    """
    Validate the complete merged AutoREACTER reaction library.

    Raises
    ------
    TypeError
        If the reaction library is not a dictionary.

    ValueError
        If one or more reaction definitions are invalid.
    """
    if not isinstance(reactions, dict):
        raise TypeError(
            "Reaction library must be a dictionary."
        )

    errors: list[str] = []

    for reaction_name, reaction in reactions.items():

        if not isinstance(reaction, dict):
            errors.append(
                f"{reaction_name}: reaction entry must be "
                "a dictionary"
            )
            continue

        errors.extend(
            _validate_reaction_smarts(
                reaction_name,
                reaction,
            )
        )

    if errors:
        message = "\n".join(
            f"  - {error}"
            for error in errors
        )

        raise ValueError(
            "Reaction library validation failed:\n"
            f"{message}"
        )


def build_reaction(rxn_smarts: str):
    """
    Create an RDKit ChemicalReaction from reaction SMARTS.

    AutoREACTER currently supports reactions containing exactly
    two reactant templates.

    Parameters
    ----------
    rxn_smarts : str
        Reaction SMARTS string.

    Returns
    -------
    rdChemReactions.ChemicalReaction
        Parsed RDKit reaction object.

    Raises
    ------
    ValueError
        If the SMARTS is empty, invalid, or does not contain exactly
        two reactant templates.
    """
    if (
        not isinstance(rxn_smarts, str)
        or not rxn_smarts.strip()
    ):
        raise ValueError(
            "Reaction SMARTS must be a non-empty string."
        )

    try:
        rxn = rdChemReactions.ReactionFromSmarts(
            rxn_smarts
        )

    except Exception as error:
        raise ValueError(
            f"Could not parse reaction SMARTS: {error}"
        ) from error

    if rxn is None:
        raise ValueError(
            "RDKit could not parse reaction SMARTS: "
            f"{rxn_smarts}"
        )

    num_reactants = rxn.GetNumReactantTemplates()

    if num_reactants != 2:
        raise ValueError(
            "AutoREACTER currently requires exactly two "
            f"reactant templates; found {num_reactants}."
        )

    if rxn.GetNumProductTemplates() == 0:
        raise ValueError(
            "Reaction SMARTS must contain at least one "
            "product template."
        )

    return rxn


def _clear_atom_maps(mol):
    """
    Remove existing atom-map numbers from an RDKit molecule.

    Reaction mapping in AutoREACTER is defined by the reaction SMARTS,
    not by map numbers that may already exist in the input SMILES.
    """
    for atom in mol.GetAtoms():
        atom.SetAtomMapNum(0)


def _build_molecule(smiles: str):
    """
    Build an RDKit molecule from SMILES and add explicit hydrogens.

    Existing atom-map numbers from the input SMILES are cleared because
    AutoREACTER derives reaction mapping from the reaction SMARTS.
    """
    if (
        not isinstance(smiles, str)
        or not smiles.strip()
    ):
        raise ValueError(
            "Reactant SMILES must be a non-empty string."
        )

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(
            "RDKit could not parse reactant SMILES: "
            f"{smiles}"
        )

    mol = Chem.AddHs(mol)

    _clear_atom_maps(mol)

    return mol


def build_reactants(
    reactant_smiles_1: str,
    reactant_smiles_2: str | None = None,
):
    """
    Build two independent RDKit reactant molecules.

    Explicit hydrogens are added to both molecules.

    If reactant_smiles_2 is None, a second independent molecule is
    constructed from reactant_smiles_1. This is important for
    self-reactions because A + A must use two separate RDKit Mol
    objects.

    Returns
    -------
    tuple
        (mol_reactant_1, mol_reactant_2)
    """
    if reactant_smiles_2 is None:
        reactant_smiles_2 = reactant_smiles_1

    mol_reactant_1 = _build_molecule(
        reactant_smiles_1
    )

    mol_reactant_2 = _build_molecule(
        reactant_smiles_2
    )

    return (
        mol_reactant_1,
        mol_reactant_2,
    )


def reaction_tuples(
    same_reactants: bool,
    mol_reactant_1,
    mol_reactant_2,
):
    """
    Generate possible reactant orderings.

    Self-reaction
    -------------
        A + A

    Different reactants
    -------------------
        A + B
        B + A

    Fresh molecule copies are generated for every attempt so atom maps
    applied during one attempt cannot contaminate another.
    """
    if same_reactants:
        return [
            (
                Chem.Mol(mol_reactant_1),
                Chem.Mol(mol_reactant_2),
            )
        ]

    return [
        (
            Chem.Mol(mol_reactant_1),
            Chem.Mol(mol_reactant_2),
        ),
        (
            Chem.Mol(mol_reactant_2),
            Chem.Mol(mol_reactant_1),
        ),
    ]


def map_reactant_atoms(
    rxn_smarts: str,
    same_reactants: bool,
    reactant_smiles_1: str,
    reactant_smiles_2: str | None = None,
):
    """
    Match reaction SMARTS to reactants and transfer atom-map numbers.

    Not every atom is required to be mapped. Only atoms explicitly
    mapped in the SMARTS receive atom-map numbers.

    AutoREACTER requires reaction mapping itself to be valid before this
    function is called:

        - mapped atoms must have unique map numbers;
        - mapped atoms must correspond 1:1 between reactants/products;
        - initiator atom maps :1 and :2 must exist;
        - :1 and :2 must form the product initiator bond.

    For different reactants, both reactant orderings are attempted:

        A + B
        B + A

    Parameters
    ----------
    rxn_smarts : str
        Reaction SMARTS string.

    same_reactants : bool
        True for A + A reactions.

    reactant_smiles_1 : str
        SMILES of the first reactant.

    reactant_smiles_2 : str, optional
        SMILES of the second reactant.

    Returns
    -------
    tuple
        (
            [(mapped_reactant_1, mapped_reactant_2)],
            products,
        )

    Raises
    ------
    ValueError
        If no valid SMARTS match or product can be generated.
    """
    if (
        not same_reactants
        and not reactant_smiles_2
    ):
        raise ValueError(
            "reactant_smiles_2 is required when "
            "same_reactants=False."
        )

    rxn = build_reaction(
        rxn_smarts
    )

    mol_reactant_1, mol_reactant_2 = (
        build_reactants(
            reactant_smiles_1,
            reactant_smiles_1
            if same_reactants
            else reactant_smiles_2,
        )
    )

    smarts_reactant_1 = (
        rxn.GetReactantTemplate(0)
    )

    smarts_reactant_2 = (
        rxn.GetReactantTemplate(1)
    )

    matched_any_order = False

    for mol_1, mol_2 in reaction_tuples(
        same_reactants,
        mol_reactant_1,
        mol_reactant_2,
    ):

        matches_1 = mol_1.GetSubstructMatches(
            smarts_reactant_1
        )

        matches_2 = mol_2.GetSubstructMatches(
            smarts_reactant_2
        )

        if not matches_1 or not matches_2:
            continue

        matched_any_order = True

        # Current AutoREACTER behavior:
        # use the first matching reactive site.
        match_1 = matches_1[0]
        match_2 = matches_2[0]

        smart_mapping(
            mol_1,
            smarts_reactant_1,
            match_1,
        )

        smart_mapping(
            mol_2,
            smarts_reactant_2,
            match_2,
        )

        products = rxn.RunReactants(
            (
                mol_1,
                mol_2,
            )
        )

        if not products:
            # SMARTS matched this ordering, but RDKit could
            # not generate a product. Try the next ordering.
            continue

        print(
            "[INFO] Reaction SMARTS matched reactants."
        )

        print(
            "[INFO] Atom mapping applied successfully."
        )

        print(
            "[INFO] Reaction produced products."
        )

        return (
            [(mol_1, mol_2)],
            products,
        )

    if not matched_any_order:
        raise ValueError(
            "No valid substructure match was found for the "
            "supplied reactants and reaction SMARTS."
        )

    raise ValueError(
        "Reaction SMARTS matched the supplied reactants, "
        "but RDKit could not generate products for any "
        "valid reactant ordering."
    )

# Test the mapping function with reaction definitions

In [ ]:
validate_reactions(reactions_definitions)

print("[INFO] Reaction library validation passed.")


for reaction_name, test in reactions_definitions.items():

    print(f"\n[INFO] Testing reaction: {reaction_name}")

    rxn_smarts = test["reaction"]
    same_reactants = test["same_reactants"]

    reactant_smiles_1 = sample_smiles[0]

    if same_reactants:
        reactant_smiles_2 = None
    else:
        reactant_smiles_2 = sample_smiles[1]

    mapped_reactants, products = map_reactant_atoms(
        rxn_smarts=rxn_smarts,
        same_reactants=same_reactants,
        reactant_smiles_1=reactant_smiles_1,
        reactant_smiles_2=reactant_smiles_2,
    )

    print(
        f"[INFO] Reaction produced "
        f"{len(products)} product set(s)."
    )


    reactant_molecules = [
        mol
        for pair in mapped_reactants
        for mol in pair
    ]

    highlight_atoms = [
        [
            atom.GetIdx()
            for atom in mol.GetAtoms()
            if atom.GetAtomMapNum() != 0
        ]
        for mol in reactant_molecules
    ]

    img = Draw.MolsToGridImage(
        reactant_molecules,
        molsPerRow=2,
        subImgSize=(600, 600),
        highlightAtomLists=highlight_atoms,
        legends=[
            f"{reaction_name} - Reactant {i + 1}"
            for i in range(len(reactant_molecules))
        ],
    )

    display(img)

# Visualize products

In [ ]:

for product_set in products:
    img = Draw.MolsToGridImage(
        product_set,
        molsPerRow=2,
        subImgSize=(900, 900),
    )
    display(img)

# (Optional) Multi-stage reaction testing

Chains reactions: the product of stage N becomes a reactant of stage N+1. Works for radical, clean (e.g. epoxy-amine), and condensation (byproduct lost) chemistry.

Rules these cells rely on:
- Reactants carry **explicit H atoms**, so product templates must **not** contain H counts (`[C:1]`, not `[CH2:1]`). RDKit adds the template H count on top of the existing H atoms and the valence breaks.
- A **hydrogen that moves** (e.g. N-H to O-H in epoxy ring opening) must be mapped on both sides as `[#1:n]`.
- **Leaving groups** (e.g. water in amidation) are left **unmapped**. RDKit deletes unmapped template atoms, and the library validator needs every mapped atom on both sides.
- RDKit silently fills under-valent atoms with implicit H, so **radical centres** are set after the reaction via `radical_maps`.
- Atom props such as `old_mapno` leak between stages, so they are cleared before each one.
- `RunReactants` returns symmetric duplicates, so products are deduplicated by canonical SMILES.
- Prefer `X` (total connections) over `D` in group SMARTS if you ever use implicit-H molecules. `D` only behaves as written with explicit H.

Optional tester-only keys in a reaction dict (not verified against AutoREACTER's schema):

| key | default | meaning |
|---|---|---|
| `radical_maps` | none | map numbers of product atoms that carry a radical |
| `radical_delta` | `0` | expected change in total radicals (so clean reactions are asserted radical-free) |
| `expected_loss` | optional | atoms lost when `delete_atom` is True, e.g. `{"heavy": 1, "H": 2}` for water. If omitted, any loss is accepted but atoms must not be gained and at least one must be lost. When `delete_atom` is False, zero loss is always required. |
| `initiator_atom_maps` | `(1, 2)` | atoms that must be bonded in the product |
| `validate_initiator_bond` | `True` | set False to skip that check |

In [ ]:
_STALE_PROPS = ("old_mapno", "react_atom_idx", "react_idx")


def clear_reaction_props(mol):
    m = Chem.Mol(mol)
    for atom in m.GetAtoms():
        for key in _STALE_PROPS:
            if atom.HasProp(key):
                atom.ClearProp(key)
    return m


def mark_radicals(product, radical_maps):
    for atom in product.GetAtoms():
        if atom.HasProp("old_mapno") and atom.GetIntProp("old_mapno") in radical_maps:
            atom.SetNoImplicit(True)
            atom.SetNumRadicalElectrons(1)


def atom_counts(mol):
    return {
        "heavy": sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() > 1),
        "H": sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 1),
        "radicals": sum(a.GetNumRadicalElectrons() for a in mol.GetAtoms()),
    }


def lint_reaction(name, reaction):
    """Static warnings about a reaction entry (no molecules needed)."""
    out = []
    rxn = rdChemReactions.ReactionFromSmarts(reaction["reaction"])

    for t in range(rxn.GetNumProductTemplates()):
        for atom in rxn.GetProductTemplate(t).GetAtoms():
            if atom.GetNumExplicitHs() > 0:
                out.append(
                    f"{name}: product atom :{atom.GetAtomMapNum()} ({atom.GetSymbol()}) has an H count; "
                    "remove it when reactants carry explicit H atoms "
                    "(map a moving H atom as [#1:n] instead)"
                )

    r_maps = {a.GetAtomMapNum() for t in range(rxn.GetNumReactantTemplates())
              for a in rxn.GetReactantTemplate(t).GetAtoms() if a.GetAtomMapNum()}
    p_maps = {a.GetAtomMapNum() for t in range(rxn.GetNumProductTemplates())
              for a in rxn.GetProductTemplate(t).GetAtoms() if a.GetAtomMapNum()}
    if r_maps != p_maps:
        out.append(f"{name}: map numbers only on one side: {sorted(r_maps ^ p_maps)} "
                   "(leaving-group atoms should be left unmapped)")

    missing = set(reaction.get("radical_maps", ())) - p_maps
    if missing:
        out.append(f"{name}: radical_maps {sorted(missing)} not present in product template")
    return out


def check_product(products, reaction, counts_in):
    """products: one Mol or the full list of Mols returned for one product set."""
    if isinstance(products, Chem.Mol):
        products = [products]
    out = {k: sum(atom_counts(p)[k] for p in products) for k in ("heavy", "H", "radicals")}
    lost = {"heavy": counts_in["heavy"] - out["heavy"], "H": counts_in["H"] - out["H"]}
    problems = []

    if not reaction.get("delete_atom", False):
        for k in ("heavy", "H"):
            if lost[k] != 0:
                problems.append(f"{k} atoms lost {lost[k]}, expected 0 (delete_atom is False)")
    else:
        expected = reaction.get("expected_loss")
        if expected is not None:
            for k in ("heavy", "H"):
                if lost[k] != expected.get(k, 0):
                    problems.append(f"{k} atoms lost {lost[k]}, expected {expected.get(k, 0)}")
        elif lost["heavy"] < 0 or lost["H"] < 0:
            problems.append("delete_atom is True but the products have more atoms than the reactants")
        elif lost["heavy"] == 0 and lost["H"] == 0:
            problems.append("delete_atom is True but no atoms were lost")

    delta = reaction.get("radical_delta", 0)
    if out["radicals"] != counts_in["radicals"] + delta:
        problems.append(
            f"radicals {counts_in['radicals']} -> {out['radicals']} (expected delta {delta:+d})"
        )

    m1, m2 = reaction.get("initiator_atom_maps", (1, 2))
    where = {}
    for i, p in enumerate(products):
        for a in p.GetAtoms():
            if a.HasProp("old_mapno"):
                where[a.GetIntProp("old_mapno")] = (i, a.GetIdx())
    if m1 in where and m2 in where:
        if reaction.get("validate_initiator_bond", True):
            (i1, a1), (i2, a2) = where[m1], where[m2]
            if i1 != i2 or products[i1].GetBondBetweenAtoms(a1, a2) is None:
                problems.append(f"no bond between initiator maps :{m1} and :{m2}")
    else:
        problems.append(f"initiator maps :{m1}/:{m2} not found in product")
    return problems, lost


def process_products(raw_products, reaction, counts_in):
    """
    Sanitize every molecule of every product set, mark radicals (if any), dedupe, and check.
    The FIRST product template is the 'main' product (used for the next stage);
    any further templates are kept as `byproducts` and counted in the atom balance.
    """
    radical_maps = set(reaction.get("radical_maps", ()))
    unique, seen, failed, last_error = [], set(), 0, None
    for product_set in raw_products:
        mols = list(product_set)
        for m in mols:
            mark_radicals(m, radical_maps)
        try:
            for m in mols:
                Chem.SanitizeMol(m)
        except Exception as exc:
            failed, last_error = failed + 1, str(exc)
            continue
        problems, lost = check_product(mols, reaction, counts_in)  # needs old_mapno
        for m in mols:
            for a in m.GetAtoms():
                a.SetAtomMapNum(0)
        smiles = [Chem.MolToSmiles(Chem.RemoveHs(m)) for m in mols]
        key = ".".join(smiles)
        if key not in seen:
            seen.add(key)
            unique.append({"mol": mols[0], "byproducts": mols[1:], "smiles": smiles[0],
                           "all_smiles": key, "problems": problems, "lost": lost})
    return {"raw": len(raw_products), "failed": failed,
            "last_error": last_error, "unique": unique}


def run_stage(reaction, reactants):
    """reactants: explicit-H mols in reaction-template order."""
    rxn = rdChemReactions.ReactionFromSmarts(reaction["reaction"])
    counts_in = {k: sum(atom_counts(m)[k] for m in reactants)
                 for k in ("heavy", "H", "radicals")}
    raw = rxn.RunReactants(tuple(clear_reaction_props(m) for m in reactants))
    return process_products(raw, reaction, counts_in)


def groups_present(mol, fg_definitions):
    found = {}
    for d in fg_definitions.values():
        n = len(mol.GetSubstructMatches(Chem.MolFromSmarts(d["smarts_1"])))
        if n:
            found[d["group_name"]] = n
    return found

In [ ]:
stage_1_name = reaction_name      # default: last reaction tested above; set to any key of reactions_definitions
stage_1 = reactions_definitions[stage_1_name]

# regenerate reactants/products for the SELECTED reaction (the test loop above only keeps the last one)
mapped_reactants, products = map_reactant_atoms(
    rxn_smarts=stage_1["reaction"],
    same_reactants=stage_1["same_reactants"],
    reactant_smiles_1=sample_smiles[0],
    reactant_smiles_2=None if stage_1["same_reactants"] else sample_smiles[1],
)

for msg in lint_reaction(stage_1_name, stage_1):
    print("[LINT]", msg)

counts_in = {k: sum(atom_counts(m)[k] for m in mapped_reactants[0])
             for k in ("heavy", "H", "radicals")}
stage_1_result = process_products(products, stage_1, counts_in)
stage_1_products = stage_1_result["unique"]

print(f"[INFO] {stage_1_name}: {stage_1_result['raw']} raw sets -> {len(stage_1_products)} unique product(s), "
      f"{stage_1_result['failed']} failed sanitization")
if not stage_1_products:
    print(f"[ERROR] No product survived sanitization. Last RDKit error: {stage_1_result['last_error']}")

for i, p in enumerate(stage_1_products, start=1):
    print(f"  product {i}: checks {'OK' if not p['problems'] else '; '.join(p['problems'])}")
    print(f"    SMILES: {p['smiles']}")
    if p["byproducts"]:
        print(f"    byproducts: {[Chem.MolToSmiles(Chem.RemoveHs(b)) for b in p['byproducts']]}")
    print(f"    groups present: {groups_present(p['mol'], monomers)}")

if stage_1_products:
    display(Draw.MolsToGridImage(
        [Chem.RemoveHs(p["mol"]) for p in stage_1_products],
        molsPerRow=1, subImgSize=(900, 600),
        legends=[f"stage 1 product {i}" for i in range(1, len(stage_1_products) + 1)],
    ))

# (Optional) Stage 2

The cells above are a complete single-stage run. Use this section only if you want to test a second stage. If you don't, leave `stage_2_reactions` empty and Run All simply skips it.

## What to fill in (next cell)

Use the same dict format as the default cells:

- **`stage_2_fgs`**: functionality definitions for groups that stage 2 needs but `monomers` doesn't define yet (e.g. a radical chain end, or a newly formed secondary amine). Use `{}` if everything is already defined.
- **`stage_2_reactions`**: reaction definitions, same format as `reactions_definitions`.

## What happens

1. The stage-2 definitions are validated with the same rules as the default ones.
2. Every stage-1 product is tried against every stage-2 reaction. The partner is either a sample monomer or another copy of the product.
3. Each candidate is actually run and checked (atoms conserved, expected radical change, initiator bond formed), and you get a yes/no answer to "is there a stage-2 reaction for at least one product?"

## Notes

- For radical stages, add `radical_maps` and `radical_delta` to the reaction dict, as **lists** (not sets) so the JSON printout works.
- For a third stage, call `find_next_stage` again with the stage-2 products and a new reaction dict.

In [ ]:
stage_2_fgs = {
    'secondary_amine_group': {
        'functionality_type': 'mono',
        'smarts_1': '[NX3;H1;!$(NC=O)]([#6])[#6]',
        'group_name': 'secondary_amine',
        'comments': 'Formed in stage 1'
    },
}

stage_2_reactions = {
    'Epoxy + secondary amine': {
        'same_reactants': False,
        'reactant_1': 'epoxide',
        'reactant_2': 'secondary_amine',
        'product': 'tertiary_amine',
        'delete_atom': False,
        'reaction': (
            '[CH2:1]1[O:3][CH:4]1.'
            '[NX3;H1;!$(NC=O):2]-[#1:5]'
            '>>'
            '[N:2]-[C:1]-[C:4]-[O:3]-[#1:5]'
        ),
    },
}

In [ ]:
def validate_fg_definitions(fg_definitions):
    """Same rules as the monomer functionality validation cell."""
    for funct in fg_definitions.values():
        functionality_type = funct.get("functionality_type")
        if functionality_type is None:
            raise ValueError("Functionality type is missing.")
        if functionality_type not in ["di_identical", "di_different", "mono", "vinyl"]:
            raise ValueError(f"Functionality type '{functionality_type}' is invalid.")
        if "smarts_1" not in funct:
            raise ValueError("Missing 'smarts_1' must be provided.")
        smart_1 = funct.get("smarts_1")
        if Chem.MolFromSmarts(smart_1) is None:
            raise ValueError(f"'smarts_1' '{smart_1}' is invalid.")
        if functionality_type == "di_different":
            smart_2 = funct.get("smarts_2")
            if smart_2 is None:
                raise ValueError("Missing 'smarts_2' for 'di_different' functionality type.")
            if Chem.MolFromSmarts(smart_2) is None:
                raise ValueError(f"'smarts_2' '{smart_2}' is invalid.")
        group_name = funct.get("group_name")
        if not isinstance(group_name, str) or not group_name.strip():
            raise ValueError(f"'group_name' '{group_name}' is invalid.")
        print(f"{group_name} definitions are valid.")


def validate_reaction_entries(reactions):
    """Same checks as the reaction validation cell, for any reaction dict."""
    for name, r in reactions.items():
        print(f"\nValidating reaction: {name}")
        if r.get("same_reactants") is None:
            raise ValueError(f"{name}: 'same_reactants' flag is missing.")
        if r.get("reactant_1") is None:
            raise ValueError(f"{name}: 'reactant_1' information is missing.")
        if not r["same_reactants"] and r.get("reactant_2") is None:
            raise ValueError(f"{name}: 'reactant_2' information is missing for different reactants.")
        if r.get("product") is None:
            raise ValueError(f"{name}: 'product' information is missing.")
        if r.get("delete_atom") is None:
            raise ValueError(f"{name}: 'delete_atom' flag is missing.")
        if r.get("reaction") is None:
            raise ValueError(f"{name}: reaction SMARTS is missing.")
        if not validate_reaction_smarts(r["reaction"]):
            raise ValueError(f"{name}: invalid reaction SMARTS.")
        is_mapped, message = are_all_atoms_mapped(r["reaction"])
        if not is_mapped:
            warnings.warn(f"{name}: {message}")
        ok, msg = validate_initiator_atom_maps(
            r["reaction"], r.get("initiator_atom_maps", (1, 2))
        )
        if not ok:
            raise ValueError(f"{name}: {msg}")
        print(msg)
        for w in lint_reaction(name, r):
            print("[LINT]", w)
    validate_reactions(reactions)
    print("\nAll stage-2 reaction definitions are valid.")


def check_references(reactions, fg_definitions):
    """Reactant group names should be defined somewhere (monomers or stage_2_fgs)."""
    defined = {d["group_name"] for d in fg_definitions.values()}
    out = []
    for name, r in reactions.items():
        for key in ("reactant_1", "reactant_2"):
            g = r.get(key)
            if g is not None and g not in defined:
                out.append(f"{name}: '{key}' = '{g}' has no functionality definition")
    return out


def find_next_stage(stage_products, reactions, pool):
    """
    Try every reaction on every stage product. Partner = a monomer from `pool`
    or another copy of the product (only two copies of the product for
    same_reactants=True). Templates are matched first, then the reaction is
    actually run and checked.
    """
    hits = []
    for p_idx, p in enumerate(stage_products, start=1):
        candidates = {"<product>": p["mol"], **pool}
        for rname, reaction in reactions.items():
            rxn = rdChemReactions.ReactionFromSmarts(reaction["reaction"])
            t0, t1 = rxn.GetReactantTemplate(0), rxn.GetReactantTemplate(1)
            for l0, m0 in candidates.items():
                for l1, m1 in candidates.items():
                    if "<product>" not in (l0, l1):
                        continue
                    if reaction.get("same_reactants") and (l0 != "<product>" or l1 != "<product>"):
                        continue          # self-reaction: two copies of the stage product only
                    if not (m0.HasSubstructMatch(t0) and m1.HasSubstructMatch(t1)):
                        continue
                    res = run_stage(reaction, (m0, m1))
                    hits.append({"product": p_idx, "reaction": rname,
                                 "reactants": (l0, l1), **res})
    return hits


def print_next_stage(hits):
    print(f"{'prod':<5}{'reaction':<26}{'reactant 1':<13}{'reactant 2':<13}{'raw':>4}{'uniq':>6}  checks")
    for h in hits:
        ok = h["unique"] and all(not u["problems"] for u in h["unique"])
        all_problems = dict.fromkeys(pr for u in h["unique"] for pr in u["problems"])
        status = "OK" if ok else ("no valid product" if not h["unique"] else "; ".join(all_problems))
        print(f"{h['product']:<5}{h['reaction']:<26}{h['reactants'][0]:<13}{h['reactants'][1]:<13}"
              f"{h['raw']:>4}{len(h['unique']):>6}  {status}")

In [ ]:
valid = []
has_stage_2 = bool(stage_2_reactions) and bool(stage_1_products)

if not has_stage_2:
    print("[INFO] No stage 2 defined (or no stage-1 product): single-stage run is complete.")
else:
    print("--- stage-2 functional-group definitions ---")
    validate_fg_definitions(stage_2_fgs)
    print("\n--- stage-2 reaction definitions ---")
    validate_reaction_entries(stage_2_reactions)
    for msg in check_references(stage_2_reactions, {**monomers, **stage_2_fgs}):
        print("[REF]", msg)

In [ ]:
if has_stage_2:
    all_fgs = {**monomers, **stage_2_fgs}
    for i, p in enumerate(stage_1_products, start=1):
        print(f"stage-1 product {i} groups: {groups_present(p['mol'], all_fgs)}")

    pool = {f"smiles[{i}]": Chem.AddHs(Chem.MolFromSmiles(s)) for i, s in enumerate(sample_smiles)}
    hits = find_next_stage(stage_1_products, stage_2_reactions, pool)
    print()
    print_next_stage(hits)

    valid = [h for h in hits if h["unique"] and all(not u["problems"] for u in h["unique"])]
    print(f"\nStage-2 reaction available for at least one stage-1 product: {bool(valid)} "
          f"({len(valid)} valid reaction/reactant combinations)")

In [ ]:
if valid:
    display(Draw.MolsToGridImage(
        [Chem.RemoveHs(h["unique"][0]["mol"]) for h in valid[:6]],
        molsPerRow=2, subImgSize=(700, 500),
        legends=[f"{h['reaction']}: {h['reactants'][0]} + {h['reactants'][1]}" for h in valid[:6]],
    ))